## Putting it together

A small but realistic Spring application, end to end:

```java
@SpringBootApplication   // = @Configuration + @ComponentScan + @EnableAutoConfiguration
public class App {
    public static void main(String[] args) { SpringApplication.run(App.class, args); }
}

@ConfigurationProperties(prefix = "payment")
public record PaymentProps(int maxRetries, Duration timeout) {}

@Repository
public class PaymentRepository {
    public Optional<Payment> findById(long id) { /* JPA */ }
}

@Service
public class PaymentService {
    private final PaymentRepository repository;
    private final ApplicationEventPublisher events;
    private final PaymentProps props;

    public PaymentService(PaymentRepository repository,
                          ApplicationEventPublisher events,
                          PaymentProps props) {
        this.repository = repository;
        this.events = events;
        this.props = props;
    }

    public Payment process(long id) {
        var payment = repository.findById(id).orElseThrow();
        events.publishEvent(new PaymentProcessed(id, payment.amount(), Instant.now()));
        return payment;
    }
}
```

Every annotation there is read at startup via module 07's reflection mechanics. No XML, no factory configuration, no manual wiring — you declared what the components are and what they need, and Spring constructed the graph. That is the whole of Spring Core; every later module builds on it.
